# PD01 · Explorar y preparar AGROSAVIA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 60 minutos.** El objetivo es reconocer la estructura de un DataFrame y preparar medidas y fechas conservando los valores originales.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD01_exploracion_agrosavia.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD01_exploracion_agrosavia.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 1–5 y la receta de registros en revisión de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD01"
archivos = ['agrosavia.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
suelo = pd.read_csv(RAW / "agrosavia.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD01


## 1. Inspección inicial · 10 minutos

Presentar la cantidad de filas y columnas, los nombres de las columnas, sus tipos y cinco registros. Explicar qué representa una fila y por qué la lectura inicial conserva los campos como texto.

📘 **Apoyo:** `shape`, `columns`, `dtypes`, `head` e `info`.

✅ **Comprobación:** El corte del manifiesto contiene 92.738 filas y 32 columnas. La tabla original debe conservar esa cantidad de filas durante el ejercicio.

In [2]:
from IPython.display import Markdown
fmt = lambda n: f"{n:,}".replace(",", ".")

total_original = len(suelo)
print("Filas y columnas:", suelo.shape)
print("Nombres de las columnas:", list(suelo.columns))
display(suelo.dtypes.to_frame("tipo"))
display(suelo.head(5))
suelo.info()



Filas y columnas: (92738, 32)
Nombres de las columnas: ['Secuencial', 'Fecha de Análisis', 'Departamento', 'Municipio', 'Cultivo', 'Estado', 'Tiempo de establecimiento', 'Topografia', 'Drenaje', 'Riego', 'Fertilizantes aplicados', 'pH agua:suelo', 'Materia organica', 'Fósforo Bray II', 'Azufre Fosfato monocalcico', 'Acidez Intercambiable', 'Aluminio intercambiable', 'Calcio intercambiable', 'Magnesio intercambiable', 'Potasio intercambiable', 'Sodio intercambiable', 'capacidad de intercambio cationico', 'Conductividad electrica', 'Hierro disponible olsen', 'Cobre disponible', 'Manganeso disponible Olsen', 'Zinc disponible Olsen', 'Boro disponible', 'Hierro disponible doble acido', 'Cobre disponible doble acido', 'Manganeso disponible doble acido', 'Zinc disponible doble \xa0acido']


,tipo
Secuencial,object
Fecha de Análisis,object
Departamento,object
Municipio,object
Cultivo,object
Estado,object
Tiempo de establecimiento,object
Topografia,object
Drenaje,object
Riego,object


,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,Riego,...,Conductividad electrica,Hierro disponible olsen,Cobre disponible,Manganeso disponible Olsen,Zinc disponible Olsen,Boro disponible,Hierro disponible doble acido,Cobre disponible doble acido,Manganeso disponible doble acido,Zinc disponible doble acido
0,1,7/01/2014,NARIÑO,SAN ANDRÉS DE TUMACO,No Indica,No indica,No indica,No indica,No indica,No indica,...,0.133,66.39,1.700,1.700,1.5,0.187,ND,ND,ND,ND
1,2,21/08/2014,HUILA,SANTA MARÍA,Granadilla,Establecido,De 1 a 5 años,Pendiente,Buen drenaje,No Tiene,...,2.749,250.9,3.7,29.4,28.70,2.084,ND,ND,ND,ND
2,3,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Pendiente,Mal drenaje,No Tiene,...,0.328,390,11.10,8.299,5,0.085,ND,ND,ND,ND
3,4,22/08/2014,ANTIOQUIA,LIBORINA,Maracuyá,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,...,0.171,200,4.2,4.699,1,0.229,ND,ND,ND,ND
4,5,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,...,0.323,117.9,3.8,4.600,0.700,0.139,ND,ND,ND,ND


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92738 entries, 0 to 92737
Data columns (total 32 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Secuencial                          92738 non-null  object
 1   Fecha de Análisis                   92738 non-null  object
 2   Departamento                        92738 non-null  object
 3   Municipio                           92738 non-null  object
 4   Cultivo                             92738 non-null  object
 5   Estado                              92738 non-null  object
 6   Tiempo de establecimiento           92738 non-null  object
 7   Topografia                          92738 non-null  object
 8   Drenaje                             92738 non-null  object
 9   Riego                               92738 non-null  object
 10  Fertilizantes aplicados             92738 non-null  object
 11  pH agua:suelo                       92738 non-null  ob

**Interpretación del resultado:**
Cada fila de suelo es un registro de análisis de suelo de AGROSAVIA: una muestra analizada en una fecha y un lugar (departamento y municipio), con sus resultados de laboratorio. La tabla tiene 92.738 filas y 32 columnas, que coincide con el manifiesto (92.738 filas y 32 columnas), así que la carga salió completa. Todo aparece como texto porque se leyó con dtype=str y keep_default_na=False: así pandas no adivina tipos ni convierte vacíos o ND en nulos sin que uno lo note, y los valores originales quedan intactos para decidir la limpieza paso a paso.




## 2. Diagnóstico de cinco campos · 10 minutos

Construir una tabla llamada `perfil` para `Secuencial`, `Fecha de Análisis`, `Departamento`, `Municipio` y `pH agua:suelo`. Para cada campo se registran los nulos de pandas, los textos vacíos después de retirar espacios y los marcadores `ND`. Las tres situaciones se cuentan por separado.

📘 **Apoyo:** `isna`, `.str.strip`, `.str.upper`, `.eq` y `sum`.

✅ **Comprobación:** `perfil` debe tener una fila por campo. Con `keep_default_na=False`, un texto vacío o `ND` no se convierte automáticamente en un nulo.

In [3]:
campos = ["Secuencial", "Fecha de Análisis", "Departamento", "Municipio", "pH agua:suelo"]
filas = []
for campo in campos:
    serie = suelo[campo]
    limpio = serie.str.strip()
    filas.append({
        "campo": campo,
        "nulos_pandas": int(serie.isna().sum()),
        "vacios": int(limpio.eq("").sum()),
        "nd": int(limpio.str.upper().eq("ND").sum()),
    })
perfil = pd.DataFrame(filas)
assert len(perfil) == len(campos)
display(perfil)

p = perfil.set_index("campo")
peor = (p["vacios"] + p["nd"]).idxmax()


,campo,nulos_pandas,vacios,nd
0,Secuencial,0,0,0
1,Fecha de Análisis,0,0,0
2,Departamento,0,0,0
3,Municipio,0,2,0
4,pH agua:suelo,0,0,0


**Interpretación del resultado:**
Los nulos de pandas dan 0 en los cinco campos, pero eso no quiere decir que los datos estén completos: pasa porque se leyó con keep_default_na=False, que deja los vacíos como texto y no como NaN. Por eso se contaron aparte los vacíos (tras quitar espacios) y los marcadores ND. El campo con más ausencias es Municipio, con 2 vacíos y 0 ND. Si solo hubiera usado isna(), habría concluido que no falta nada; las tres situaciones se separan porque significan cosas distintas, aunque luego todas terminen como valor ausente al convertir.


## 3. Conversión de pH y fecha · 15 minutos

Crear `ph` desde `pH agua:suelo` y `fecha_dt` desde `Fecha de Análisis`. Utilizar conversión numérica con `errors="coerce"` y fecha con formato `%d/%m/%Y`. Conservar los originales. Distinguir entradas vacías de textos no vacíos que no pudieron convertirse.

📘 **Apoyo:** `pd.to_numeric`, `.str.replace`, `pd.to_datetime`, `isna` y máscaras booleanas.

✅ **Comprobación:** Se muestran los conteos de conversiones fallidas y hasta cinco ejemplos con su valor original. No se reemplazan medidas ausentes por cero.

In [4]:
original_ph = suelo["pH agua:suelo"]
original_fecha = suelo["Fecha de Análisis"]

suelo["ph"] = pd.to_numeric(
    original_ph.str.strip().str.replace(",", ".", regex=False), errors="coerce")
suelo["fecha_dt"] = pd.to_datetime(
    original_fecha.str.strip(), format="%d/%m/%Y", errors="coerce")

vacio_ph = original_ph.str.strip().eq("")
vacio_fecha = original_fecha.str.strip().eq("")
fallo_ph = suelo["ph"].isna() & ~vacio_ph
fallo_fecha = suelo["fecha_dt"].isna() & ~vacio_fecha

resumen_conv = pd.DataFrame({
    "campo": ["pH agua:suelo", "Fecha de Análisis"],
    "vacios": [int(vacio_ph.sum()), int(vacio_fecha.sum())],
    "fallos_de_conversion": [int(fallo_ph.sum()), int(fallo_fecha.sum())],
    "ausentes_total": [int(suelo["ph"].isna().sum()), int(suelo["fecha_dt"].isna().sum())],
})
display(resumen_conv)
print("Ejemplos de pH no convertidos:")
display(suelo.loc[fallo_ph, ["Secuencial", "pH agua:suelo"]].head(5))
print("Ejemplos de fechas no interpretadas:")
display(suelo.loc[fallo_fecha, ["Secuencial", "Fecha de Análisis"]].head(5))

assert len(suelo) == total_original

ej_ph = ", ".join(f"`{v}`" for v in original_ph[fallo_ph].str.strip().value_counts().head(3).index) or "ninguno"


,campo,vacios,fallos_de_conversion,ausentes_total
0,pH agua:suelo,0,11,11
1,Fecha de Análisis,0,0,0


Ejemplos de pH no convertidos:


,Secuencial,pH agua:suelo
7282,7283,4..01
46528,46516,>10.00
46530,46518,>10.00
52903,52901,<3.80
56330,56329,<3.80


Ejemplos de fechas no interpretadas:


,Secuencial,Fecha de Análisis


**Interpretación del resultado:**

El pH quedó con 11 valores ausentes: 0 vienen de celdas vacías y 11 de textos que no se pudieron convertir (los más frecuentes: <3.80, >10.00, 4..01). En la fecha hay 0 ausentes: 0 vacías y 0 con un formato distinto a %d/%m/%Y. Separar vacío de fallo importa porque lo primero es un dato que nunca se registró y lo segundo es un dato que existe pero está mal escrito y quizá se pueda recuperar. Se conservaron las columnas originales para poder ver qué había escrito, y no se reemplazó ninguna medida ausente por cero, porque un pH de 0 sería un valor falso y alteraría cualquier promedio.

## 4. Registros para revisión · 15 minutos

Crear la columna booleana `en_revision`. Se marca una fila si su pH es ausente, está fuera de `[0, 14]` o su fecha no se pudo interpretar. Contar por separado cada motivo y el total de filas marcadas. Conservar todas las observaciones.

📘 **Apoyo:** `between`, `isna`, `|`, `~`, `.loc` y `value_counts`.

✅ **Comprobación:** Las filas marcadas y no marcadas deben sumar el total original. Una misma fila puede tener varios motivos; por eso la suma de los motivos no necesariamente coincide con el total marcado.

In [5]:
motivo_ph_ausente = suelo["ph"].isna()
motivo_ph_fuera = suelo["ph"].notna() & ~suelo["ph"].between(0, 14)
motivo_fecha = suelo["fecha_dt"].isna()

suelo["en_revision"] = motivo_ph_ausente | motivo_ph_fuera | motivo_fecha

conteo_motivos = pd.Series({
    "pH ausente": int(motivo_ph_ausente.sum()),
    "pH fuera de [0, 14]": int(motivo_ph_fuera.sum()),
    "fecha no interpretable": int(motivo_fecha.sum()),
}, name="filas")
display(conteo_motivos.to_frame())
display(suelo["en_revision"].value_counts())

marcadas = int(suelo["en_revision"].sum())
no_marcadas = int((~suelo["en_revision"]).sum())
assert marcadas + no_marcadas == total_original
assert len(suelo) == total_original

display(Markdown(f"""**Interpretación del resultado:** Se marcaron {fmt(marcadas)} filas para revisión y {fmt(no_marcadas)} quedaron sin marcar; juntas suman {fmt(marcadas + no_marcadas)}, igual que el total original, así que no se perdió ningún registro. Por motivos: {fmt(int(motivo_ph_ausente.sum()))} por pH ausente, {fmt(int(motivo_ph_fuera.sum()))} por pH fuera de [0, 14] y {fmt(int(motivo_fecha.sum()))} por fecha no interpretable. La suma de los motivos ({fmt(int(conteo_motivos.sum()))}) no coincide con las filas marcadas ({fmt(marcadas)}) porque una misma fila puede tener varios motivos a la vez. Marcar en lugar de borrar deja el registro disponible para corregirlo o excluirlo de forma consciente."""))

,filas
pH ausente,11
"pH fuera de [0, 14]",0
fecha no interpretable,0


en_revision
False    92727
True        11
Name: count, dtype: int64

**Interpretación del resultado:** Se marcaron 11 filas para revisión y 92.727 quedaron sin marcar; juntas suman 92.738, igual que el total original, así que no se perdió ningún registro. Por motivos: 11 por pH ausente, 0 por pH fuera de [0, 14] y 0 por fecha no interpretable. La suma de los motivos (11) no coincide con las filas marcadas (11) porque una misma fila puede tener varios motivos a la vez. Marcar en lugar de borrar deja el registro disponible para corregirlo o excluirlo de forma consciente.

**Interpretación del resultado:**

Se marcaron 11 filas para revisión y 92.727 quedaron sin marcar; juntas suman 92.738, igual que el total original, así que no se perdió ningún registro. Por motivos: 11 por pH ausente, 0 por pH fuera de [0, 14] y 0 por fecha no interpretable. La suma de los motivos (11) no coincide con las filas marcadas (11) porque una misma fila puede tener varios motivos a la vez. Marcar en lugar de borrar deja el registro disponible para corregirlo o excluirlo de forma consciente.


## 5. Evidencias y conclusiones · 10 minutos

Construir `controles` con las columnas `control` y `cantidad`: total de entrada, pH sin número, pH fuera del intervalo, fecha no interpretable y total de filas en revisión. Exportar a `OUT / "controles_agrosavia.csv"` sin índice. Redactar dos hallazgos apoyados en los conteos.

📘 **Apoyo:** `pd.DataFrame`, `to_csv(index=False)` y las tablas de las actividades anteriores.

✅ **Comprobación:** El archivo se genera dentro de la ruta `OUT`. Cada hallazgo menciona un conteo y explica su significado o limitación.

In [6]:
controles = pd.DataFrame({
    "control": ["total de entrada", "pH sin número", "pH fuera del intervalo",
                "fecha no interpretable", "total de filas en revisión"],
    "cantidad": [total_original, int(suelo["ph"].isna().sum()), int(motivo_ph_fuera.sum()),
                 int(suelo["fecha_dt"].isna().sum()), int(suelo["en_revision"].sum())],
})
display(controles)
controles.to_csv(OUT / "controles_agrosavia.csv", index=False)
print("Archivo generado:", (OUT / "controles_agrosavia.csv").is_file())

pct = 100 * marcadas / total_original
principal = conteo_motivos.idxmax()
display(Markdown(f"""**Hallazgo 1.** Hay {fmt(marcadas)} filas en revisión de {fmt(total_original)}, es decir {pct:.1f} % de la tabla, y el motivo más frecuente es *{principal}* ({fmt(int(conteo_motivos.max()))} filas). Antes de analizar pH o series de tiempo hay que decidir qué hacer con ellas. Limitación: el control solo detecta lo que no se convierte o cae fuera de rango; un pH mal digitado pero dentro de [0, 14] pasaría sin ser detectado.

**Hallazgo 2.** Hay {fmt(int(motivo_ph_fuera.sum()))} pH fuera de [0, 14] y {fmt(int(motivo_fecha.sum()))} fechas sin interpretar con `%d/%m/%Y`. Un pH fuera de ese intervalo es físicamente imposible y sugiere error de digitación o de unidades, mientras que las fechas fallidas pueden deberse a otro formato y quizá se recuperen. Aun así, el intervalo es solo un control de dominio de esta práctica y no basta para dar una recomendación agronómica."""))

,control,cantidad
0,total de entrada,92738
1,pH sin número,11
2,pH fuera del intervalo,0
3,fecha no interpretable,0
4,total de filas en revisión,11


Archivo generado: True


**Hallazgo 1.** Hay 11 filas en revisión de 92.738, es decir 0.0 % de la tabla, y el motivo más frecuente es *pH ausente* (11 filas). Antes de analizar pH o series de tiempo hay que decidir qué hacer con ellas. Limitación: el control solo detecta lo que no se convierte o cae fuera de rango; un pH mal digitado pero dentro de [0, 14] pasaría sin ser detectado.

**Hallazgo 2.** Hay 0 pH fuera de [0, 14] y 0 fechas sin interpretar con `%d/%m/%Y`. Un pH fuera de ese intervalo es físicamente imposible y sugiere error de digitación o de unidades, mientras que las fechas fallidas pueden deberse a otro formato y quizá se recuperen. Aun así, el intervalo es solo un control de dominio de esta práctica y no basta para dar una recomendación agronómica.

**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## Entrega

El notebook completado se conserva como `soluciones/PD01_exploracion_agrosavia.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD01/controles_agrosavia.csv`.

✅ Deben quedar visibles el perfil, los controles de conversión, la conciliación de filas y dos hallazgos. El notebook debe ejecutar todas las celdas en orden después de reiniciar el kernel.

⚠️ El intervalo de pH es un control de dominio para esta práctica. No determina por sí solo una recomendación agronómica. La frecuencia de análisis de suelo no representa automáticamente la distribución de los suelos de todo el territorio.